# Iniciando o Spark

In [ ]:
## Bloco de codigo para instalar versao especifica dos pacotes
!pip install pyspark

In [ ]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Trusted_base_recarga") \
    .getOrCreate()

# Importando bibliotecas

In [ ]:
import os
import pytz
import datetime
from datetime import datetime
#from pyspark.sql.types import *
#from pyspark.sql.functions import count, avg
#import sys
#import numpy as np
#from datetime import datetime
#from pyspark.sql import SQLContext
#from datetime import timedelta
#from datetime import date
#from dateutil.relativedelta import relativedelta
#from pyspark.sql.functions import udf, lpad, translate

# Funções auxiliares e variáveis

In [ ]:
# Função de log
def log():
    return datetime.now().strftime('%Y-%m-%d %H:%M:%S') + " >>>"

# Timestamp de processamento (com hora/minuto/segundo)
agora = datetime.now(pytz.timezone('America/Sao_Paulo'))
dthproc = agora.strftime("%Y%m%d%H%M%S")

# Data da execução (AAAAmmdd)
PROCESS_DATE = datetime.now().strftime("%Y%m%d")

# Período de referência (AAAAmm)
REF_PERIOD = datetime.now().strftime("%Y%m")

# Buckets e nomes de saída
bucket_base = "base_recarga"
bucket_trusted = f"s3://hackathon_2025/{PROCESS_DATE}/0003_trusted/{bucket_base}"
bucket_raw = f"s3://hackathon_2025/{PROCESS_DATE}/0002_raw/{bucket_base}"
bucket_control = f"s3://hackathon_2025/{PROCESS_DATE}/0005_control/{bucket_base}"
output_trusted = f"trusted_{bucket_base}"

# Prints para conferência
print("PROCESS_DATE:", PROCESS_DATE)
print("REF_PERIOD:", REF_PERIOD)
print("dthproc:", dthproc)
print("bucket_trusted:", bucket_trusted)
print("bucket_raw:", bucket_raw)
print("bucket_control:", bucket_control)


#  Leitura da camada Raw

In [ ]:
path_raw = os.path.join(bucket_raw, bucket_base)

parquet_files = [path_raw for f in os.listdir(bucket_raw) if f.endswith('.parquet')]
df_raw_recarga= spark.read.parquet(*parquet_files, header=True, inferSchema=True)
df_raw_recarga.createOrReplaceTempView("raw_base_recarga")

print(log(), "Registros na Raw:", df_raw_recarga.count())
df_raw_recarga.show(5, truncate=False)

### Montando as dimensões para enriquecer a base

In [ ]:
PATH_DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO": f"{path_raw}/BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": f"{path_raw}/BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": f"{path_raw}/BI_DIM_INSTITUICAO.csv",
    "PLATAFORMA": f"{path_raw}/BI_DIM_PLATAFORMA.csv",
    "PROMOCAO": f"{path_raw}/BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": f"{path_raw}/BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": f"{path_raw}/BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": f"{path_raw}/BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": f"{path_raw}/BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": f"{path_raw}/BI_DIM_TIPO_RECARGA.csv",
}

DIMENSOES_RECARGA = {
    "CANAL_AQUISICAO_CREDITO": "BI_DIM_CANAL_AQUISICAO_CREDITO.csv",
    "FORMA_PAGAMENTO": "BI_DIM_FORMA_PAGAMENTO.csv",
    "INSTITUICAO": "BI_DIM_INSTITUICAO.csv",
    "PLANO_PRECO": "BI_DIM_PLANO_PRECO.csv",
    "PLATAFORMA": "BI_DIM_PLATAFORMA.csv",
    "PROMOCAO_CREDITO": "BI_DIM_PROMOCAO_CREDITO.csv",
    "STATUS_PLATAFORMA": "BI_DIM_STATUS_PLATAFORMA.csv",
    "TECNOLOGIA": "BI_DIM_TECNOLOGIA.csv",
    "TIPO_CREDITO": "BI_DIM_TIPO_CREDITO.csv",
    "TIPO_INSERCAO": "BI_DIM_TIPO_INSERCAO.csv",
    "TIPO_RECARGA": "BI_DIM_TIPO_RECARGA.csv",
}


dfs_dim_recarga = {}

for nome_dim, arquivo in DIMENSOES_RECARGA.items():
    path = f"{path_raw}/{arquivo}"

    dfs_dim_recarga[nome_dim] = (
        spark.read
        .option("header", True)
        .option("sep", ",")
        .option("inferSchema", True)
        .csv(path)
    )

df_CANAL_AQUISICAO_CREDITO = dfs_dim_recarga["CANAL_AQUISICAO_CREDITO"]
df_FORMA_PAGAMENTO        = dfs_dim_recarga["FORMA_PAGAMENTO"]
df_INSTITUICAO            = dfs_dim_recarga["INSTITUICAO"]
df_PLANO_PRECO            = dfs_dim_recarga["PLANO_PRECO"]
df_PLATAFORMA             = dfs_dim_recarga["PLATAFORMA"]
df_PROMOCAO_CREDITO       = dfs_dim_recarga["PROMOCAO_CREDITO"]
df_STATUS_PLATAFORMA      = dfs_dim_recarga["STATUS_PLATAFORMA"]
df_TECNOLOGIA              = dfs_dim_recarga["TECNOLOGIA"]
df_TIPO_CREDITO           = dfs_dim_recarga["TIPO_CREDITO"]
df_TIPO_INSERCAO          = dfs_dim_recarga["TIPO_INSERCAO"]
df_TIPO_RECARGA           = dfs_dim_recarga["TIPO_RECARGA"]

In [ ]:
# optamos por não inserir as dimensões nessa camada,
# para manter a granularidade e flexibilidade da base, 
# e evitar possíveis problemas de atualização das dimensões.
# As dimensões serão integradas em análises futuras na construção dos books
# garantindo que a camada Trusted permaneça o mais fiel possível à fonte original.

# Processamento tipagem para camada Trusted

In [ ]:
df_base_recarga = spark.sql("""
    SELECT
        '{dthproc}' AS ts_proc,
        '{dthproc}' AS ts_proc_partition,
        CAST(NUM_CPF AS STRING) AS NUM_CPF,
        CAST(DW_NUM_NTC AS STRING) AS DW_NUM_NTC,
        CAST(DW_NUM_CLIENTE AS STRING) AS DW_NUM_CLIENTE,
        CAST(DAT_INSERCAO_CREDITO AS STRING) AS DAT_INSERCAO_CREDITO,
        CAST(HOR_INSERCAO_CREDITO AS STRING) AS HOR_INSERCAO_CREDITO,
        CAST(COD_TECNOLOGIA_DW AS STRING) AS COD_TECNOLOGIA_DW,
        CAST(COD_CANAL_AQUISICAO AS INT) AS COD_CANAL_AQUISICAO,
        CAST(COD_TIPO_CREDITO AS STRING) AS COD_TIPO_CREDITO,
        CAST(COD_PROMOCAO AS INT) AS COD_PROMOCAO,
        CAST(VAL_CREDITO_INSERIDO AS FLOAT) AS VAL_CREDITO_INSERIDO,
        CAST(VAL_BONUS AS FLOAT) AS VAL_BONUS,
        CAST(VAL_REAL AS FLOAT) AS VAL_REAL,
        CAST(COD_PLATAFORMA_ATU AS STRING) AS COD_PLATAFORMA_ATU,
        CAST(COD_STATUS_PLATAFORMA AS STRING) AS COD_STATUS_PLATAFORMA,
        CAST(IND_METODO_PAGAMENTO AS STRING) AS IND_METODO_PAGAMENTO,
        CAST(DW_PLANO_TARIFACAO AS INT) AS DW_PLANO_TARIFACAO,
        CAST(DW_TIPO_RECARGA AS INT) AS DW_TIPO_RECARGA,
        CAST(DW_TIPO_INSERCAO AS INT) AS DW_TIPO_INSERCAO,
        CAST(DW_FORMA_PAGAMENTO AS INT) AS DW_FORMA_PAGAMENTO,
        CAST(DW_INSTITUICAO AS INT) AS DW_INSTITUICAO,
        CAST(COD_GRUPO_CARTAO AS STRING) AS COD_GRUPO_CARTAO,
        CAST(DSC_GRUPO_CARTAO_WPP AS STRING) AS DSC_GRUPO_CARTAO_WPP,
        CAST(FLAG_SOS AS INT) AS FLAG_SOS,
        CAST(VALOR_SOS AS INT) AS VALOR_SOS
    FROM raw_base_recarga
""")

df_base_recarga.createOrReplaceTempView("lake_recarga")
df_base_recarga.cache()

print(log(), "Registros Base:", df_base_recarga.count())
#df_base_recarga.printSchema()
df_base_recarga.show(5, truncate=False)

# Salvar na camada Trusted

In [ ]:
path_trusted = os.path.join(bucket_trusted, output_trusted)
print("Trusted path:", path_trusted)

df_base_recarga.write \
    .partitionBy("ts_proc_partition") \
    .mode("overwrite") \
    .option("compression", "snappy") \
    .parquet(path_trusted)

# Controle de carga

In [ ]:
df_controle = spark.sql (f"""
SELECT
    '{output_trusted}' AS name_file,
    ts_proc,
    ts_proc_partition,
    count(*) as qtd_registros
from lake_recarga
    GROUP BY 1,2,3
""")

df_controle.show()

# Controle de processamento

In [ ]:
path_control = os.path.join(bucket_control, f'tb_controle_processamento_{bucket_base}_trusted')
print("Control path:", path_control)

df_controle.write \
  .mode('append') \
  .option('compression', 'snappy') \
  .parquet(path_control)